# This might be all we need!

In [62]:
from schwingermodel import generateSchwingerHamiltonian

In [63]:
from qiskit.quantum_info import SparsePauliOp

def bitflip_projection(op: SparsePauliOp) -> SparsePauliOp:
    terms = []
    for label, coeff in zip(op.paulis.to_labels(), op.coeffs):
        new_label = label.replace("Y", "X").replace("Z", "I")
        terms.append((new_label, coeff))
    return SparsePauliOp.from_list(terms).simplify()


def apply_ix_string_to_bitstring(ix_label: str, bitstring: str) -> str:
    out = list(bitstring)
    for j, p in enumerate(ix_label):
        if p == "X":
            out[j] = "1" if out[j] == "0" else "0"
    return "".join(out)


def reachable_bitstrings_n_steps(op: SparsePauliOp, initial_bitstrings, n_steps: int):
    """
    Ignore amplitudes entirely.
    Track only which bitstrings are reachable after n_steps applications
    of the projected operator's flip masks.
    """
    if isinstance(initial_bitstrings, str):
        current = {initial_bitstrings}
    else:
        current = set(initial_bitstrings)

    labels = op.paulis.to_labels()

    for _ in range(n_steps):
        nxt = set()
        for b in current:
            for label in labels:
                nxt.add(apply_ix_string_to_bitstring(label, b))
        current = nxt

    return current

In [64]:
# Example parameters
N = 4
x = 1.0
lam = 0.5
l0 = 0.0
m_lat = 1.0
g = 1.0

In [ ]:
N = 30
H = generateSchwingerHamiltonian(N, x, lam, l0, m_lat, g)
H_flip = bitflip_projection(H)

initial_state = '01' * (N//2)


# Post-processing

# does not work yet

In [66]:
import numpy as np
import scipy.linalg
from qiskit.quantum_info import SparsePauliOp


def apply_pauli_to_bitstring(pauli_label: str, bitstring: str):
    """
    Apply a Pauli string to a computational basis bitstring.

    Returns
    -------
    phase : complex
        The phase acquired.
    out_bitstring : str
        The resulting computational basis bitstring.

    Conventions
    -----------
    - `pauli_label` is a Qiskit Pauli label string, e.g. "IXYZ".
    - `bitstring` is a standard computational basis bitstring, e.g. "0101".
    - We assume the displayed bitstring ordering matches the Pauli label ordering.
    """
    bits = list(bitstring)
    phase = 1.0 + 0.0j

    for i, p in enumerate(pauli_label):
        b = bits[i]

        if p == 'I':
            continue
        elif p == 'X':
            bits[i] = '1' if b == '0' else '0'
        elif p == 'Y':
            # Y|0> = i|1>,  Y|1> = -i|0>
            bits[i] = '1' if b == '0' else '0'
            phase *= 1j if b == '0' else -1j
        elif p == 'Z':
            # Z|0> = |0>, Z|1> = -|1>
            if b == '1':
                phase *= -1
        else:
            raise ValueError(f"Invalid Pauli character: {p}")

    return phase, ''.join(bits)


def project_hamiltonian_to_bitstring_subspace(H: SparsePauliOp, bitstrings):
    """
    Project a SparsePauliOp Hamiltonian into the subspace spanned by the
    given computational-basis bitstrings.

    Parameters
    ----------
    H : SparsePauliOp
        Full Hamiltonian on N qubits.
    bitstrings : sequence[str]
        Bitstrings defining the subspace basis.

    Returns
    -------
    H_sub : np.ndarray
        Dense projected Hamiltonian matrix in the bitstring basis.
    basis : list[str]
        The ordered basis actually used.
    """
    # Remove duplicates but preserve order
    basis = list(dict.fromkeys(bitstrings))
    m = len(basis)
    if m == 0:
        raise ValueError("bitstrings must not be empty")

    n = len(basis[0])
    if any(len(b) != n for b in basis):
        raise ValueError("All bitstrings must have the same length")

    basis_index = {b: i for i, b in enumerate(basis)}

    H_sub = np.zeros((m, m), dtype=complex)

    labels = H.paulis.to_labels()
    coeffs = H.coeffs

    # Matrix element construction:
    # For each column basis state |b_j>, act with each Pauli term.
    # If it lands on another basis state |b_i> inside the chosen subspace,
    # add coeff * phase to H_sub[i, j].
    for j, b_in in enumerate(basis):
        for label, coeff in zip(labels, coeffs):
            phase, b_out = apply_pauli_to_bitstring(label, b_in)
            i = basis_index.get(b_out)
            if i is not None:
                H_sub[i, j] += coeff * phase

    return H_sub, basis


def diagonalize_projected_hamiltonian(H_sub):
    """
    Diagonalize a projected Hamiltonian matrix.

    Uses scipy.linalg.eigh, assuming H_sub is Hermitian.
    """
    evals, evecs = scipy.linalg.eigh(H_sub)
    return evals, evecs

In [67]:
reachable = reachable_bitstrings_n_steps(H_flip, initial_state, 1)
bitstrings = sorted(reachable)

In [68]:
H_full = generateSchwingerHamiltonian(N, x, lam, l0, m_lat, g)

H_sub, basis = project_hamiltonian_to_bitstring_subspace(H_full, bitstrings)

evals, evecs = diagonalize_projected_hamiltonian(H_sub)

print("Basis:")
print(basis)

print("\nProjected Hamiltonian:")
print(H_sub)

print("\nEigenvalues:")
print(evals)

Basis:
['0011010101010101', '0100110101010101', '0101001101010101', '0101010011010101', '0101010100110101', '0101010101001101', '0101010101010011', '0101010101010101', '0101010101010110', '0101010101011001', '0101010101100101', '0101010110010101', '0101011001010101', '0101100101010101', '0110010101010101', '1001010101010101']

Projected Hamiltonian:
[[-11.+0.j   0.+0.j   0.+0.j   0.+0.j   0.+0.j   0.+0.j   0.+0.j   1.+0.j
    0.+0.j   0.+0.j   0.+0.j   0.+0.j   0.+0.j   0.+0.j   0.+0.j   0.+0.j]
 [  0.+0.j -11.+0.j   0.+0.j   0.+0.j   0.+0.j   0.+0.j   0.+0.j   1.+0.j
    0.+0.j   0.+0.j   0.+0.j   0.+0.j   0.+0.j   0.+0.j   0.+0.j   0.+0.j]
 [  0.+0.j   0.+0.j -11.+0.j   0.+0.j   0.+0.j   0.+0.j   0.+0.j   1.+0.j
    0.+0.j   0.+0.j   0.+0.j   0.+0.j   0.+0.j   0.+0.j   0.+0.j   0.+0.j]
 [  0.+0.j   0.+0.j   0.+0.j -11.+0.j   0.+0.j   0.+0.j   0.+0.j   1.+0.j
    0.+0.j   0.+0.j   0.+0.j   0.+0.j   0.+0.j   0.+0.j   0.+0.j   0.+0.j]
 [  0.+0.j   0.+0.j   0.+0.j   0.+0.j -11.+0.j   0.+

In [69]:
import numpy as np
from scipy.sparse.linalg import eigsh

def exact_ground_state_energy(H):
    """
    Compute the exact ground-state energy of a SparsePauliOp Hamiltonian
    using sparse diagonalization.

    Parameters
    ----------
    H : SparsePauliOp

    Returns
    -------
    float
        Ground-state energy.
    """
    # Convert to sparse matrix (CSR format)
    H_sparse = H.to_matrix(sparse=True)

    # Compute smallest algebraic eigenvalue
    evals, _ = eigsh(H_sparse, k=1, which='SA')
    return float(evals[0].real)

In [70]:
H_full = generateSchwingerHamiltonian(N, x, lam, l0, m_lat, g)
E0 = exact_ground_state_energy(H_full)
print("Exact ground state energy:", E0)
print('vs. approximated via Krylov: ', min(evals))

Exact ground state energy: -18.722105495497562
vs. approximated via Krylov:  -18.10977222864642
